# Обучение детектора архитектуры EAST

Пример для Manuscript OCR 0.1.13 на датасете [handwritten_essay](https://huggingface.co/datasets/sherstpasha/handwritten_essay): русские рукописные школьные сочинения с разметкой слов.

Данные берём из двух источников:

- изображения — [Mendeley Data, версия 1](https://data.mendeley.com/datasets/vs44v8r3nf/1), DOI `10.17632/vs44v8r3nf.1`;
- COCO-разметку слов — Hugging Face `sherstpasha/handwritten_essay`.

Изображения опубликованы под лицензией CC BY-NC 3.0.

## 1. Установка


In [1]:
%pip install "manuscript-ocr[dev]==0.1.13" huggingface-hub requests



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Загрузка данных

В архиве Mendeley изображения вложены в папки: `train/0/0.png`, `test/0/0.png` и т. д. Разметка Hugging Face использует плоские имена с инвертированными префиксами:

- `train/0/0.png` → `train/test_0_0.png` из `train_coco.json`;
- `test/0/0.png` → `test/train_0_0.png` из `test_coco.json`.

Слово `test` в имени не означает, что страницу нужно переносить в тестовый раздел. Раздел определяется COCO-файлом и исходной папкой. Ниже создаём отдельные плоские папки, сохраняя оригиналы в `raw`.

Сохраняем это разделение: 28 страниц для обучения, 131 для валидации. Отдельного валидационного набора здесь нет, поэтому используем `test`; после подбора параметров он уже не будет независимым тестом.


In [2]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile
import hashlib
import json
from huggingface_hub import hf_hub_download

DATA_DIR = Path("datasets/handwritten_essay")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Прямая ссылка на архив Mendeley версии 1: API списка файлов не требуется.
archive_url = (
    "https://data.mendeley.com/public-files/datasets/vs44v8r3nf/files/"
    "3a105c2e-02e9-4fc5-91e2-4d828ceb2d90/file_downloaded"
)
archive_sha256 = "274bd7e4be4c63f68d8f39d88ca89414bf30858d27b49731d58cc161481391ff"
archive_path = DATA_DIR / "handwritten_essay.zip"
if not archive_path.exists():
    urlretrieve(archive_url, archive_path)

# Проверяем полноту скачанного архива перед распаковкой.
with archive_path.open("rb") as stream:
    digest = hashlib.file_digest(stream, "sha256").hexdigest()
assert digest == archive_sha256, "Архив повреждён: удалите его и скачайте снова"

with ZipFile(archive_path) as zf:
    zf.extractall(DATA_DIR / "raw")

# Hugging Face содержит разметку, а не изображения.
for filename in ("train_coco.json", "test_coco.json"):
    hf_hub_download(
        repo_id="sherstpasha/handwritten_essay",
        repo_type="dataset", filename=filename, local_dir=DATA_DIR,
    )


/Users/user/manuscript-ocr/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HTTPError: HTTP Error 403: Forbidden

## 3. Подготовка и проверка разметки

Создадим плоские папки изображений с именами из COCO. Для каждой страницы проверим размеры из разметки: это помогает обнаружить перепутанные разделы или файлы до обучения.


In [ ]:
import shutil
from PIL import Image

RAW_DIR = DATA_DIR / "raw"
IMAGES_DIR = DATA_DIR / "images"
for split in ("train", "test"):
    annotations = json.loads((DATA_DIR / f"{split}_coco.json").read_text())
    destination = IMAGES_DIR / split
    destination.mkdir(parents=True, exist_ok=True)
    for item in annotations["images"]:
        # Префикс имени инвертирован; исходный split берём из COCO-файла.
        prefix, essay_id, page_id = Path(item["file_name"]).stem.split("_")
        expected_prefix = "test" if split == "train" else "train"
        assert prefix == expected_prefix, item["file_name"]
        source = RAW_DIR / split / essay_id / f"{page_id}.png"
        assert source.is_file(), f"Не найдено изображение: {source}"
        with Image.open(source) as original:
            assert original.size == (item["width"], item["height"]), source
        shutil.copy2(source, destination / item["file_name"])
    print(f"{split}: {len(annotations['images'])} страниц, "
          f"{len(annotations['annotations'])} слов; все изображения проверены")

# Покажем страницу и исходные полигоны слов до обучения.
from PIL import Image
from IPython.display import display
from manuscript.data import Page, TextRegion
from manuscript.utils.visualization import visualize_page

sample = json.loads((DATA_DIR / "train_coco.json").read_text())
info = sample["images"][0]
image = Image.open(IMAGES_DIR / "train" / info["file_name"]).convert("RGB")
page = Page(image_width=image.width, image_height=image.height)
for annotation in sample["annotations"]:
    if annotation["image_id"] == info["id"]:
        coordinates = annotation["segmentation"][0]
        region = TextRegion(type="word", polygon=list(zip(coordinates[::2], coordinates[1::2])))
        region.parent = page

display(image)
display(visualize_page(image, page, numbers_levels=[], max_size=1440, upscale=True))


## 4. Обучение

Одна эпоха позволяет проверить весь процесс; для обучения полезной модели увеличьте `EPOCHS` и следите за метриками валидации.

`target_size` задаёт размер входа обучения, `batch_size` — число страниц в пакете. `detection_level="word"` записывает в конфиг, что модель выдаёт слова; этот параметр не преобразует разметку строк в слова.


In [ ]:
import torch
from manuscript.detectors import EAST

EXP_ROOT = Path("experiments")
MODEL_NAME = "east_handwritten_essay"
EXP_DIR = EXP_ROOT / MODEL_NAME
EPOCHS = 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = EAST.train(
    train_images=str(IMAGES_DIR / "train"),
    train_anns=str(DATA_DIR / "train_coco.json"),
    val_images=str(IMAGES_DIR / "test"),
    val_anns=str(DATA_DIR / "test_coco.json"),
    experiment_root=str(EXP_ROOT),
    model_name=MODEL_NAME,
    backbone_name="resnet50",
    pretrained_backbone=True,  # Начинаем с предобученного backbone.
    target_size=512,
    epochs=EPOCHS,
    batch_size=1,
    num_workers=0,
    use_multiscale=False,  # Для примера используем один размер входа.
    detection_level="word",
    device=DEVICE,
)


## 5. Артефакты и загрузка модели

После обучения выполняется экспорт в ONNX. Для своего детектора нужны **два файла**: `checkpoints/best_model.onnx` и `checkpoints/best_model.json`. Веса PyTorch и состояние обучения нужны для продолжения обучения или повторного экспорта; `training_config.json` описывает параметры эксперимента.


In [ ]:
import os

for directory, _, filenames in os.walk(EXP_DIR):
    for filename in sorted(filenames):
        path = Path(directory) / filename
        print(path.relative_to(EXP_DIR), f"({path.stat().st_size / 1024:.1f} КБ)")

# Передаём оба обязательных артефакта явно.
detector = EAST(
    weights=str(EXP_DIR / "checkpoints/best_model.onnx"),
    config=str(EXP_DIR / "checkpoints/best_model.json"),
    device="cpu",
)


## 6. Проверка на странице валидации

Покажем исходный снимок и найденные области. После одной эпохи качество ещё не показательно.


In [ ]:
validation = json.loads((DATA_DIR / "test_coco.json").read_text())
image_path = IMAGES_DIR / "test" / validation["images"][0]["file_name"]
image = Image.open(image_path).convert("RGB")
display(image)
page = detector.predict(image)
print(f"Найдено областей: {len(page.children)}")
display(visualize_page(image, page, numbers_levels=[], max_size=1440, upscale=True))
